# Formative Assignment: Advanced Linear Algebra (PCA)

This notebook implements **Principal Component Analysis (PCA) from scratch** on an Africanized dataset.

**Dataset:** African socio-economic indicators (30 countries, 10 columns). Themes: *economic activity* (GDP per capita, agriculture share of GDP) and *population pressure* (population density, urbanisation).

It satisfies the brief: **7+ columns**, **missing/NaN values**, and **at least one non-numeric column** (`Country`).

> **Library note:** All PCA mathematics (standardisation, covariance, eigendecomposition, sorting, projection) uses **numpy only** — no `sklearn`. `pandas` is used solely to *load and clean* the raw CSV (separate the non-numeric column, impute NaN) and `matplotlib` solely to *visualise*, since neither can be done in numpy.

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd            # loading & cleaning only
import matplotlib.pyplot as plt  # visualisation only

np.random.seed(42)

# Load the raw Africanized dataset
df = pd.read_csv('african_socioeconomic.csv')
print('Raw shape:', df.shape)
print('Missing values per column:')
print(df.isna().sum())
df.head()

: 

### Step 0: Inspect, separate the non-numeric column, and handle missing values

The brief requires a non-numeric column and NaN values. PCA operates only on numeric features, so we:
1. keep `Country` aside as **labels** (used later for plotting),
2. impute missing numeric values with the **column mean** (a simple, leakage-free choice for this small dataset).

In [ ]:
# Separate the non-numeric label column
labels = df['Country'].values                      # non-numeric column kept as labels
features_df = df.drop(columns=['Country'])         # numeric feature matrix
feature_names = features_df.columns.tolist()

# Impute NaN with column means, then convert to a pure numpy array
data = features_df.values.astype(float)            # may contain NaN
col_means = np.nanmean(data, axis=0)               # mean ignoring NaN
nan_mask = np.isnan(data)
data[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

print('Clean numeric matrix shape:', data.shape)
print('Remaining NaN:', int(np.isnan(data).sum()))
print('Features:', feature_names)
data[:5]

### Step 1: Load and Standardize the Data
Before applying PCA we standardize so every feature has mean 0 and standard deviation 1. This implements the z-score formula  $z = \dfrac{x - \mu}{\sigma}$  (per the assignment image), using numpy only.

In [ ]:
# Step 1: Standardize the data (numpy only) -> z = (x - mean) / std
mean = np.mean(data, axis=0)
std  = np.std(data, axis=0)            # population std (ddof=0)
standardized_data = (data - mean) / std

print('Means after standardization (~0):', np.round(standardized_data.mean(axis=0), 6))
print('Stds  after standardization (~1):', np.round(standardized_data.std(axis=0), 6))
standardized_data[:5]  # first few rows of standardized data

### Step 3: Calculate the Covariance Matrix
The covariance matrix shows how every pair of features varies together. It is the core object PCA decomposes.

In [ ]:
# Step 3: Covariance matrix (numpy only). rowvar=False -> columns are variables.
cov_matrix = np.cov(standardized_data, rowvar=False)
print('Covariance matrix shape:', cov_matrix.shape)
np.round(cov_matrix, 3)

**Why compute a covariance matrix? (max 5 lines)**

The covariance matrix measures how every pair of features varies together, summarising the linear relationships and redundancy in the data in one symmetric matrix. PCA needs it for two reasons: (1) its **eigenvectors** give the orthogonal directions of maximum variance — the principal components; and (2) its **eigenvalues** quantify how much variance each direction carries, which lets us rank components and decide how many to keep.

### Step 4: Perform Eigendecomposition
Eigendecomposition of the covariance matrix yields the eigenvalues and eigenvectors used by PCA. Because the covariance matrix is symmetric we use `np.linalg.eigh`, which returns real values and is numerically stable for symmetric matrices.

In [ ]:
# Step 4: Eigendecomposition (numpy only)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
print('Eigenvalues:\n', np.round(eigenvalues, 4))
print('\nEigenvectors (columns) shape:', eigenvectors.shape)
eigenvalues, eigenvectors

### Step 5: Sort Principal Components
`eigh` returns eigenvalues in *ascending* order, so we sort them **descending**: the larger the eigenvalue, the more variance (information) that component holds. We also compute the explained-variance ratio, used in Step 6 to choose the number of components.

In [ ]:
# Step 5: Sort eigenvalues (and matching eigenvectors) in descending order
sorted_indices = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues  = eigenvalues[sorted_indices]
sorted_eigenvectors = eigenvectors[:, sorted_indices]

# Explained variance ratio + cumulative
explained_variance_ratio = sorted_eigenvalues / np.sum(sorted_eigenvalues)
cumulative_variance = np.cumsum(explained_variance_ratio)

print('Sorted eigenvalues:        ', np.round(sorted_eigenvalues, 4))
print('Explained variance ratio:  ', np.round(explained_variance_ratio, 4))
print('Cumulative variance:       ', np.round(cumulative_variance, 4))
sorted_eigenvectors

### Step 6: Project Data onto Principal Components
We **dynamically** select the smallest number of components whose **cumulative explained variance** reaches a threshold (90%), then project the standardized data onto those components.

In [ ]:
# Step 6: Dynamically choose number of components, then project
variance_threshold = 0.90
num_components = int(np.argmax(cumulative_variance >= variance_threshold) + 1)
print(f'Components to retain >= {variance_threshold:.0%} variance: {num_components}')
print(f'Variance actually retained: {cumulative_variance[num_components-1]:.2%}')

# Projection: standardized data onto the top-k eigenvectors
reduced_data = standardized_data @ sorted_eigenvectors[:, :num_components]
print('Projected (reduced) data shape:', reduced_data.shape)
reduced_data[:5]

### Step 7: Output the Reduced Data

In [ ]:
# Step 7: Output the reduced data
print(f'Original Data Shape: {standardized_data.shape}')
print(f'Reduced  Data Shape: {reduced_data.shape}')
reduced_data[:5]

### Step 8: Visualize Before and After PCA
Left: two original standardized features. Centre: the data in the new PC1–PC2 space. Right: a scree / cumulative-variance plot justifying the component count.

In [ ]:
# Step 8: Visualize Before and After PCA
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

# (A) Original data: first two standardized features
ax[0].scatter(standardized_data[:, 0], standardized_data[:, 1], c='steelblue', s=60, edgecolor='k')
ax[0].set_xlabel(feature_names[0]); ax[0].set_ylabel(feature_names[1])
ax[0].set_title('Original data (first 2 standardized features)')
ax[0].grid(alpha=0.3)

# (B) Reduced data after PCA: PC1 vs PC2
ax[1].scatter(reduced_data[:, 0], reduced_data[:, 1], c='crimson', s=60, edgecolor='k')
for i, name in enumerate(labels):          # annotate countries
    ax[1].annotate(name, (reduced_data[i, 0], reduced_data[i, 1]), fontsize=6, alpha=0.7)
ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2')
ax[1].set_title(f'After PCA ({num_components} components kept)')
ax[1].grid(alpha=0.3)

# (C) Scree + cumulative variance
comps = np.arange(1, len(explained_variance_ratio) + 1)
ax[2].bar(comps, explained_variance_ratio, color='lightgray', edgecolor='k', label='Individual')
ax[2].plot(comps, cumulative_variance, 'o-', color='darkorange', label='Cumulative')
ax[2].axhline(variance_threshold, ls='--', color='green', label=f'{variance_threshold:.0%} threshold')
ax[2].axvline(num_components, ls=':', color='purple')
ax[2].set_xlabel('Principal component'); ax[2].set_ylabel('Explained variance ratio')
ax[2].set_title('Scree plot'); ax[2].legend(); ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Answers (max 5 lines each):**

**1. Interpret the before/after visual.**
On the left, the two raw standardized features spread along both axes with visible correlation (high-GDP countries cluster apart from low-GDP ones). After PCA the same countries are re-expressed on PC1–PC2, the axes of greatest variance: most of the spread now lies along PC1, and the axes are uncorrelated. PCA has simply *rotated* the coordinate system so that information is concentrated into fewer directions while the relative separation between countries is preserved.

**2. Why this many components, and the tradeoffs.**
I kept the smallest number of components reaching ~90% cumulative explained variance. The tradeoff is **compression vs. fidelity**: more components retain more detail but give back less dimensionality reduction (less noise removal, larger storage/compute); fewer components compress harder but discard more variance and may merge distinct countries. The 90% rule keeps the dominant economic/demographic structure while dropping low-variance noise.

**3. What information is lost (economic activity / population pressure).**
The dropped low-variance components are discarded, so we lose the *fine, country-specific deviations* from the dominant continental pattern — e.g. a state with very high **population pressure** but atypically low **economic activity** (agriculture-heavy, low GDP) gets pulled toward the main trend. We also lose the exact original feature values and the ability to flag such outliers; only the strongest combined signals survive.

### Task 3: Optimize for performance & handle large datasets

For tall datasets (many rows) the covariance + `eigh` route scales with the number of **features**, not rows, so it stays cheap. For very wide data, an **SVD-based** PCA avoids forming the covariance matrix explicitly and is more numerically stable. Below we (a) wrap PCA in a reusable numpy-only function and (b) benchmark covariance-eigendecomposition vs SVD on a large synthetic matrix — results match to numerical precision.

In [ ]:
# Task 3: numpy-only PCA function (reusable) + SVD variant
def pca_cov(X, k):
    Xc = (X - X.mean(0)) / X.std(0)
    C = np.cov(Xc, rowvar=False)
    w, V = np.linalg.eigh(C)
    idx = np.argsort(w)[::-1]
    return Xc @ V[:, idx[:k]]

def pca_svd(X, k):
    Xc = (X - X.mean(0)) / X.std(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)  # no covariance matrix formed
    return Xc @ Vt[:k].T

# Verify both give the same subspace on our real data (sign-invariant)
a = pca_cov(data, num_components)
b = pca_svd(data, num_components)
print('Cov vs SVD agree (abs):', np.allclose(np.abs(a), np.abs(b), atol=1e-6))
a[:3]

In [ ]:
# Benchmark on a large synthetic dataset: 200,000 rows x 50 features
import time
big = np.random.randn(200_000, 50)

t0 = time.perf_counter(); _ = pca_cov(big, 5); t_cov = time.perf_counter() - t0
t0 = time.perf_counter(); _ = pca_svd(big, 5); t_svd = time.perf_counter() - t0

print(f'Dataset: {big.shape[0]:,} rows x {big.shape[1]} features')
print(f'Covariance + eigh : {t_cov:.3f} s')
print(f'SVD               : {t_svd:.3f} s')
print('-> Covariance route is fastest for tall/skinny data (cost set by #features);')
print('   SVD is preferred when features are very many or higher numerical stability is needed.')